**Підготовка даних**

In [ ]:
import pandas as pd

# Додаємо low_memory=False, щоб прибрати попередження про типи даних
df = pd.read_csv('survey_results_public.csv', low_memory=False)
schema = pd.read_csv('survey_results_schema.csv')

# Налаштовуємо відображення всіх колонок
pd.set_option('display.max_columns', None)

print("Дані успішно завантажені без попереджень.")

Дані успішно завантажені без попереджень.


**Завдання 1. Підрахунок загальної кількості респондентів**

In [ ]:
# Використовуємо унікальні ідентифікатори респондентів для точного підрахунку
total_respondents = df['ResponseId'].nunique()
print(f"Загальна кількість респондентів: {total_respondents}")

Загальна кількість респондентів: 49191


**Завдання 2. Аналіз повноти відповідей респондентів**

In [ ]:
# Отримуємо назви питань зі схеми та перевіряємо їх наявність у колонках датасету
target_questions = set(schema['qname']).intersection(set(df.columns))

# Шукаємо респондентів, у яких заповнені абсолютно всі ці колонки
complete_responses = df.dropna(subset=list(target_questions))
print(f"Кількість респондентів, що відповіли на всі питання: {len(complete_responses)}")

Кількість респондентів, що відповіли на всі питання: 0


**Завдання 3. Статистичний аналіз досвіду роботи респондентів**

In [ ]:
# Обчислюємо ключові метрики для колонки WorkExp
mean_exp = df['WorkExp'].mean()
median_exp = df['WorkExp'].median()
mode_exp = df['WorkExp'].mode()[0]

print(f"Середній досвід: {mean_exp:.1f} років")
print(f"Медіанний досвід: {median_exp} років")
print(f"Найчастіший досвід (мода): {mode_exp} років")

Середній досвід: 13.4 років
Медіанний досвід: 10.0 років
Найчастіший досвід (мода): 10.0 років


**Завдання 4. Аналіз віддаленої роботи**

In [ ]:
# Рахуємо кількість респондентів, які працюють віддалено
remote_workers_count = df[df['RemoteWork'] == 'Remote'].shape[0]
print(f"Кількість віддалених працівників: {remote_workers_count}")

Кількість віддалених працівників: 10931


**Завдання 5. Визначення популярності Python**

In [ ]:
# Перевіряємо наявність 'Python' у рядках із мовами програмування
python_users = df['LanguageHaveWorkedWith'].str.contains('Python', na=False).sum()
percentage = (python_users / df['ResponseId'].nunique()) * 100

print(f"Відсоток респондентів, які працюють з Python: {percentage:.2f}%")

Відсоток респондентів, які працюють з Python: 37.54%


**Завдання 6. Аналіз шляхів навчання програмуванню**

In [ ]:
# Визначаємо кількість тих, хто вказував навчання через онлайн-курси
online_learners = df['LearnCode'].str.contains('Online Courses', na=False).sum()
print(f"Кількість респондентів, що навчалися через онлайн-курси: {online_learners}")

Кількість респондентів, що навчалися через онлайн-курси: 10973


**Завдання 7. Географічний аналіз компенсації Python-розробників**

In [ ]:
# Фільтруємо групу Python-програмістів
python_devs = df[df['LanguageHaveWorkedWith'].str.contains('Python', na=False)]

# Групуємо по країнах та рахуємо середню і медіанну зарплату
salary_stats = python_devs.groupby('Country')['ConvertedCompYearly'].agg(['mean', 'median'])
salary_stats.head(10)

,mean,median
Country,,
Afghanistan,22328.666667,1000.0
Albania,47217.600000,50000.0
Algeria,20187.285714,7088.0
Andorra,226103.500000,226103.5
Angola,NaN,NaN
Antigua and Barbuda,1.000000,1.0
Argentina,47080.207792,40000.0
Armenia,19093.400000,2075.0
Australia,118091.410423,97514.0


**Завдання 8. Аналіз освіти найбільш оплачуваних спеціалістів**

In [ ]:
# Сортуємо за компенсацією та вибираємо топ-5
top_paid = df.sort_values(by='ConvertedCompYearly', ascending=False).head(5)

# Виводимо рівень освіти для цих спеціалістів
top_paid[['EdLevel', 'ConvertedCompYearly']]

,EdLevel,ConvertedCompYearly
34267,"Associate degree (A.A., A.S., etc.)",50000000.0
28700,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",33552715.0
43143,"Associate degree (A.A., A.S., etc.)",18387548.0
35353,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",15430267.0
45971,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",13921760.0


**Завдання 9. Аналіз популярності Python по вікових категоріях**

*Тут ми спочатку групуємо людей за віком, а потім всередині кожної групи рахуємо, скільки людей знають Python.*

In [ ]:
# Створюємо допоміжну колонку, яка показує, чи знає людина Python (True/False)
df['IsPythonUser'] = df['LanguageHaveWorkedWith'].str.contains('Python', na=False)

# Групуємо за віком та рахуємо середнє значення для True/False (це і буде відсоток)
age_python_stats = df.groupby('Age')['IsPythonUser'].mean() * 100

# Перетворюємо на DataFrame для гарного вигляду
age_python_stats = age_python_stats.reset_index()
age_python_stats.columns = ['Вікова категорія', 'Відсоток Python-розробників (%)']

# Виводимо результат
age_python_stats

,Вікова категорія,Відсоток Python-розробників (%)
0,18-24 years old,40.000000
1,25-34 years old,36.939282
2,35-44 years old,36.719281
3,45-54 years old,38.629482
4,55-64 years old,37.242955
5,65 years or older,31.634820
6,Prefer not to say,31.216931


**Завдання 10. Аналіз індустрій серед високооплачуваних віддалених працівників**

*Тут ми використовуємо метод quantile(0.75), щоб знайти той самий поріг зарплати, вище якого знаходяться топ-25% багатіїв.*

In [ ]:
# 1. Знаходимо поріг 75-го перцентиля для компенсації
salary_threshold = df['ConvertedCompYearly'].quantile(0.75)

# 2. Фільтруємо респондентів: зарплата вище порогу ТА віддалена робота
top_remote_workers = df[
    (df['ConvertedCompYearly'] >= salary_threshold) &
    (df['RemoteWork'] == 'Remote')
]

# 3. Рахуємо частоту індустрій у цій групі
# Користуємося value_counts(), він автоматично сортує від найпопулярніших
top_industries = top_remote_workers['Industry'].value_counts().reset_index()
top_industries.columns = ['Індустрія', 'Кількість респондентів']

# Виводимо результат
top_industries

,Індустрія,Кількість респондентів
0,Software Development,1186
1,Fintech,190
2,Healthcare,188
3,Other:,176
4,"Internet, Telecomm or Information Services",138
5,Banking/Financial Services,88
6,Government,78
7,Media & Advertising Services,75
8,Retail and Consumer Services,65
9,"Transportation, or Supply Chain",63
